# ForecastIQ — Notebook 2B: FULL M5 Statistical Benchmark

**Research question:** Do statistical forecasting results observed on our balanced 400-series pilot hold across the entire 30,490-series bottom-level dataset?

Seven candidate models: Historical Average, Naive, Seasonal Naive (7), SES, Croston, SBA, TSB. Each model's smoothing parameter is selected from {0.1, 0.2, 0.3} **on past validation periods only**; the globally selected, demand-category selected, and per-series strategies are also chosen using validation only. The final 28-day test is untouched until scoring.

Full-dataset runs can be CPU-intensive; test `MAX_SERIES=200` first. Only one forecast origin is used for the final test—future work will add multi-origin backtests.

In [21]:
# Shared scientifically validated ForecastIQ framework
import sys
from pathlib import Path
_project = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(_project) not in sys.path:
    sys.path.insert(0, str(_project))
from src.experiments.config import ExperimentConfig, save_manifest
EXPERIMENT_CONFIG = ExperimentConfig()
EXPERIMENT_NAME = '02B_full_statistical_benchmark'
MANIFEST = save_manifest(_project / 'results' / 'manifests' / (EXPERIMENT_NAME + '.json'), EXPERIMENT_NAME, EXPERIMENT_CONFIG, sources=[_project / 'data' / 'raw' / 'sales_train_validation.csv'])
print('Experiment manifest:', MANIFEST['experiment_id'])


Experiment manifest: 02B_full_statistical_benchmark-20261101T102951Z


## 1 — Load full data and set strict time boundaries

In [22]:
from pathlib import Path
import numpy as np, pandas as pd, time
from IPython.display import display
PROJECT=Path.cwd()
if not (PROJECT/'data').exists() and (PROJECT.parent/'data').exists(): PROJECT=PROJECT.parent
OUT=PROJECT/'data'/'processed'; OUT.mkdir(parents=True,exist_ok=True)
MAX_SERIES=None # set 200 for smoke test; None = all 30,490
H=28; FOLDS=3; ALPHAS=(.1,.2,.3); UNDER=5.; OVER=1.
sales=pd.read_csv(OUT/'m5_full_sales.csv')
meta=pd.read_csv(OUT/'m5_full_metadata.csv')
if MAX_SERIES is not None:
    sales=sales.head(MAX_SERIES).copy()
meta=meta[['series_id','demand_type']]
panel=sales.merge(meta,on='series_id',how='left',validate='one_to_one',suffixes=('','_meta'))
days=sorted([c for c in panel if c.startswith('d_') and c[2:].isdigit()],key=lambda c:int(c[2:]))
Y=panel[days].to_numpy(dtype=np.float32)
assert np.isfinite(Y).all() and (Y>=0).all()
N,T=Y.shape; TEST=T-H
assert T>H*(FOLDS+1)+112
origins=[TEST-H*i for i in range(FOLDS,0,-1)]
print('Rows:',N,'Days:',T,'Validation origins:',origins,'Test origin:',TEST)
display(panel.demand_type.value_counts())

Rows: 30490 Days: 1913 Validation origins: [1801, 1829, 1857] Test origin: 1885


demand_type
Intermittent            22179
Lumpy                    5561
Smooth                   1895
Erratic                   854
Insufficient history        1
Name: count, dtype: int64

## 2 — Vectorized forecasting functions
All models return **expected daily demand**. Croston and SBA estimate a constant rate; TSB updates sale occurrence probability on every historical day. SES uses every historical day including zeros. `np.nan` or negative forecasts are not allowed.

In [23]:
from src.forecasting.statistical_models import forecast_matrix
def forecast_all(history, model, alpha=.2, horizon=H):
    kwargs = {'alpha': alpha} if model in ('SES','Croston','SBA','TSB') else {}
    return forecast_matrix(model, history, horizon, **kwargs).astype(np.float32)


## 3 — Validation: per-series alpha tuning and model selection
Each model competes on **three chronological fixed-origin 28-day validation blocks**. Within a model family, alpha is selected per series by mean validation MAE. For Average/Naive/SeasonalNaive alpha is not applicable. This does not train models on the final 28 test days.

In [24]:
start=time.time()
cv_by_model={}; selected_alpha={}
for model in MODELS:
    choices=[None] if model in ('Average','Naive','SeasonalNaive') else list(ALPHAS)
    scores=[]
    for alpha in choices:
        errors=[]
        for origin in origins:
            p=forecast_all(Y[:,:origin],model,alpha if alpha is not None else .2)
            errors.append(np.abs(p-Y[:,origin:origin+H]).mean(axis=1))
        scores.append(np.stack(errors).mean(axis=0))
    m=np.stack(scores)
    k=m.argmin(axis=0)
    cv_by_model[model]=m[k,np.arange(N)]
    selected_alpha[model]=np.array([choices[j] if choices[j] is not None else np.nan for j in k])
    print(f'{model:14s} mean CV MAE={cv_by_model[model].mean():.4f}')
cv_mat=np.column_stack([cv_by_model[m] for m in MODELS]);best_index=cv_mat.argmin(axis=1)
print('CV elapsed sec:',round(time.time()-start,1))

Average        mean CV MAE=1.1506
Naive          mean CV MAE=1.3534
SeasonalNaive  mean CV MAE=1.2052
SES            mean CV MAE=1.0054
Croston        mean CV MAE=1.0786
SBA            mean CV MAE=1.0535
TSB            mean CV MAE=1.0035
CV elapsed sec: 62.5


## 4 — Test predictions, with alpha selected by validation
Tuned alpha predictions are assembled by parameter group for computational efficiency. The test is **scored only after every model and parameter choice is frozen**.

In [25]:
test_truth=Y[:,TEST:TEST+H]
test_preds={}
for model in MODELS:
    chosen=selected_alpha[model]
    if model in ('Average','Naive','SeasonalNaive'):
        p=forecast_all(Y[:,:TEST],model)
    else:
        p=np.zeros((N,H),dtype=np.float32)
        for a in ALPHAS:
            ix=np.flatnonzero(np.isclose(chosen,a))
            if len(ix):p[ix]=forecast_all(Y[ix,:TEST],model,a)
    assert p.shape==test_truth.shape and np.isfinite(p).all()
    test_preds[model]=np.maximum(p,0)
# Save compact per-series model summaries; optional forecasts are saved separately.
cv_table=pd.DataFrame({'series_id':panel.series_id,'demand_type':panel.demand_type})
for m in MODELS:cv_table['cv_'+m]=cv_by_model[m]
cv_table['per_series_model']=np.array(MODELS)[best_index]
cv_table.to_csv(OUT/'phase2b_full_cv_scores.csv',index=False)
print('Full-model test predictions created')

Full-model test predictions created


## 5 — Validation-only global/category/per-series selection
The same 7 tuned candidate model forecasts are reused for every strategy. The model families are chosen using validation MAE rather than test data.

In [26]:
global_model=MODELS[int(cv_mat.mean(axis=0).argmin())]
category_model={}
for cat in panel.demand_type.dropna().unique():
    mask=(panel.demand_type.to_numpy()==cat)
    category_model[cat]=MODELS[int(cv_mat[mask].mean(axis=0).argmin())]
strategies={'Global':np.repeat(global_model,N),
            'PerDemandType':np.array([category_model[c] for c in panel.demand_type]),
            'PerSeries':np.array(MODELS)[best_index]}
print('Global:',global_model,'By category:',category_model)
for name,choice in strategies.items():
    p=np.empty((N,H),dtype=np.float32)
    for m in MODELS:
        mask=choice==m
        p[mask]=test_preds[m][mask]
    test_preds['Strategy_'+name]=p

Global: TSB By category: {'Intermittent': 'TSB', 'Lumpy': 'TSB', 'Erratic': 'TSB', 'Smooth': 'TSB', 'Insufficient history': 'Naive'}


## 6 — MAE, bias, WAPE, MASE, RMSSE and inventory-cost proxy
MASE/RMSSE use **training-only** nonseasonal naive scales. WAPE is ratio of aggregate absolute error to aggregate absolute actual demand, not a mean of product-specific ratios. Inventory-cost proxy is **hypothetical daily under/overprediction penalty** (not actual replenishment simulation).

In [ ]:
scale_mae=np.abs(np.diff(Y[:,:TEST],axis=1)).mean(axis=1)
scale_mse=(np.diff(Y[:,:TEST],axis=1)**2).mean(axis=1)
records=[];series_rows=[]
for model,p in test_preds.items():
    e=p-test_truth; ae=np.abs(e)
    sm=ae.mean(axis=1); mse=(e*e).mean(axis=1)
    cost=UNDER*np.maximum(-e,0)+OVER*np.maximum(e,0)
    records.append({'model':model,'series':N,'MAE':ae.mean(),'WAPE_pct':100*ae.sum()/test_truth.sum() if test_truth.sum()>0 else np.nan,
                    'bias':e.mean(),'mean_MASE':np.nanmean(np.divide(sm,scale_mae,out=np.full(N,np.nan),where=scale_mae>0)),
                    'mean_RMSSE':np.nanmean(np.sqrt(np.divide(mse,scale_mse,out=np.full(N,np.nan),where=scale_mse>0))),
                    'inventory_proxy':cost.sum()})
    series_rows.append(pd.DataFrame({'series_id':panel.series_id,'demand_type':panel.demand_type,'store_id':panel.store_id,'model':model,
                                    'MAE':sm,'bias':e.mean(axis=1),'cost':cost.sum(axis=1)}))
leaderboard=pd.DataFrame(records).sort_values('MAE')
per_series=pd.concat(series_rows,ignore_index=True)
by_type=per_series.groupby(['demand_type','model']).agg(series=('MAE','size'),mean_MAE=('MAE','mean'),bias=('bias','mean'),cost=('cost','sum')).reset_index()
display(leaderboard.round(4));display(by_type.round(4))
leaderboard.to_csv(OUT/'phase2b_full_leaderboard.csv',index=False)
per_series.to_csv(OUT/'phase2b_full_per_series.csv',index=False)
by_type.to_csv(OUT/'phase2b_full_by_type.csv',index=False)
print('Saved Phase 2B outputs to',OUT)

## 7 — Interpretation and limitations

- These forecasts cover all eligible **bottom-level** series, not all M5 hierarchy aggregation levels.
- Category labels were calculated before the *final* 28-day cutoff; **do not reuse them as historically known categories for earlier rolling-origin forecasts**.
- Test winners must not be used to choose model families retroactively.
- Multi-origin uncertainty estimates and official M5 WRMSSE remain future work.
- Baselines here are vectorized implementations and **must be compared against Notebook 2 on an identical small subset first**, especially Croston/TSB initialization, before treating any numerical differences as algorithmic evidence.
- 30,490-series runs may require minutes or more on CPU.